# CMR Report Generation on Colab (GPU)

This notebook runs the Cardiac Diagnostic CMR report pipeline on **Google Colab with GPU** for much faster generation.

1. **Runtime → Change runtime type → T4 GPU** (or A100 if available).
2. Run the cells below.
3. For PTB-XL you need `data/` (see [data/README.md](https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-) in the repo). Upload or mount your data, or use the sample step below.

In [13]:
# Clone the repo (skip if already cloned)
!git clone https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-.git repo_cmr
%cd repo_cmr

Cloning into 'repo_cmr'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 178 (delta 7), reused 178 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 26.55 MiB | 9.38 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/repo_cmr/repo_cmr


In [14]:
# Install dependencies and enable GPU
import os
os.environ["USE_TRANSFORMERS"] = "1"
os.environ["USE_CUDA"] = "1"  # use GPU on Colab

!pip install -q transformers torch pandas numpy scipy

In [33]:
%cd /content/repo_cmr
!python scripts/download_ptbxl_metadata.py


/content/repo_cmr
URL: https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
Output: /content/repo_cmr/data/ptbxl_database.csv

✓ Successfully downloaded metadata to: /content/repo_cmr/data/ptbxl_database.csv
  File size: 6.29 MB


In [34]:
!find data -maxdepth 3 -type f -name "*.csv"


data/ptbxl_database.csv
data/ptbxl_with_labels/ecg_with_labels.csv


In [35]:
!test -f data/ptbxl_database.csv && echo "OK, metadata present" || echo "still missing"
!head -n 3 data/ptbxl_database.csv


OK, metadata present
ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,scp_codes,heart_axis,infarction_stadium1,infarction_stadium2,validated_by,second_opinion,initial_autogenerated_report,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
1,15709.0,56.0,1,,63.0,2.0,0.0,CS-12   E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",,,,,False,False,True,," , I-V1,  ",,,,,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,,70.0,2.0,0.0,CS-12   E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,"{'NORM': 80.0, 'SBRAD': 0.0}",,,,,False,False,True,,,,,,,2,records100/00000/00002_lr,records500/00000/00002_hr


## Option A: PTB-XL + TinyLlama (fast on GPU)

Ensure `data/ptbxl_database.csv`, `data/ptbxl_with_labels/ecg_with_labels.csv`, and at least one ECG file (e.g. in `data/ptbxl_pclr_format/`) are present. Then run:

In [36]:
# Run CMR generation with TinyLlama on GPU (ecg_id 2 by default)
%cd /content/repo_cmr
!python ecg_to_cmr_report/e_to_c_llama1.py --ecg-id 2

/content/repo_cmr
Loading PTB-XL record (metadata + ECG signal)...
Traceback (most recent call last):
  File "/content/repo_cmr/ecg_to_cmr_report/e_to_c_llama1.py", line 245, in <module>
    main()
  File "/content/repo_cmr/ecg_to_cmr_report/e_to_c_llama1.py", line 217, in main
    record = load_ptbxl_record(ecg_id)
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/repo_cmr/ecg_to_cmr_report/e_to_c_llama1.py", line 92, in load_ptbxl_record
    ecg_path = _get_ecg_file_path(ecg_id)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/repo_cmr/ecg_to_cmr_report/e_to_c_llama1.py", line 53, in _get_ecg_file_path
    raise FileNotFoundError(f"ECG file missing: {path}")
FileNotFoundError: ECG file missing: /Users/tarantojeremie/Desktop/Research MIT/data/ptbxl_pclr_format/00000/00002_lr.npy


## Option B: PTB-XL + MedGemma 4B (medical LLM, GPU)

Requires Hugging Face token (accept [MedGemma terms](https://huggingface.co/google/medgemma-4b-it)). Set your token in Colab secrets or below:

In [ ]:
# Uncomment and set your HF token, or use Colab Secrets (e.g. HF_TOKEN)
# os.environ["HF_TOKEN"] = "your_hf_token_here"
!python -m ecg_to_cmr_report.e_to_c_medgemma4b --ecg-id 2 --max-tokens 128

## Data on Colab

- **Upload**: Use the Files panel to upload `ptbxl_database.csv`, `ecg_with_labels.csv`, and a sample `.npy` into `data/` and `data/ptbxl_with_labels/`, `data/ptbxl_pclr_format/` as expected by the scripts.
- **Google Drive**: Mount Drive and copy your `data/` (and optionally `MEETI/`) into the cloned repo root.

In [ ]:
# Optional: mount Google Drive and copy data
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r "/content/drive/MyDrive/Research MIT/data" .
# !cp -r "/content/drive/MyDrive/Research MIT/MEETI" .   # if using MEETI